# Husky 와이어태핑을 통한 Voltage Glitching 오류주입 공격

## 두 ChipWhisperer 장치의 역할 분리 — Lite (통신·프로그래머) + Husky (Voltage Glitch 주입)

---

### 🎯 강의 목표

이 노트북은 **ChipWhisperer 입문자를 대상**으로, 기존 Clock Glitching Wire-Tapping 시나리오를 **Voltage Glitching (크로바 글리치)** 기반의 능동적 오류주입 공격으로 고도화합니다.

- **Lite** : 정상 사용자 역할 (타겟 프로그래밍 + SimpleSerial UART 통신)
- **Husky** : 공격자 역할 (정밀 **Voltage Crowbar Glitch** 주입 + 트리거 동기화)

| 단계 | 내용 | 핵심 산출물 |
|:----:|:----|:----|
| **1단계** | 다중 장치 연결 (Lite + Husky) | `lite_scope`, `husky_scope` |
| **2단계** | Lite로 타겟 프로그래밍 및 UART 통신 채널 확보 | `target` 객체 |
| **3단계** | **상세 하드웨어 배선 가이드** (Voltage Glitch 핵심) | 배선 다이어그램 + 주의사항 |
| **4단계** | Husky Voltage Glitch 모듈 설정 | `husky_voltage_glitch_setup()` |
| **5단계** | 베이스라인 통신 테스트 | `expected_ret` 확인 |
| **6단계** | 글리치 파라미터 광범위 탐색 (3중 루프) | `vglitch_result_arr` (6단계 분류) |
| **7단계** | 통계 분석으로 최적 파라미터 도출 | `(offset, width)` 최빈값 |
| **8단계** | Bokeh 인터랙티브 시각화 | 파라미터 맵 |
| **9단계** | 장비 연결 해제 | 안전한 종료 |

> 💡 **이 노트북의 핵심 차별점**
> - **Voltage Glitching** 중심 (Clock Glitch → Voltage Crowbar Glitch)
> - **하드웨어 배선**을 극도로 상세히 다룸 (실제 공격에서 가장 중요한 부분)
> - 현실적인 "와이어태핑" 시나리오: 통신은 정상 유지하면서 **전원 라인만 tap**하여 글리치 주입


---

## 🔍 시작하기 전에 — Voltage Glitching의 특징

### 왜 Voltage Glitching인가?

- **Clock Glitching**은 클럭 라인에 글리치를 주입 → 타이밍 violation 유발
- **Voltage Glitching**은 전원 라인(Vcc/Core)을 순간적으로 단락(크로바) → 전압 강하로 명령 스킵/변조
- Voltage Glitching이 **더 강력하고 범용적**이며, 많은 실제 공격에서 사용됨
- Husky는 고성능 HP/LP MOSFET을 내장하여 강력한 Voltage Glitch 가능

### ⚠️ Voltage Glitching 하드웨어 배선의 중요성 (가장 중요!)

Voltage Glitching은 **배선 품질**에 따라 성공률이 극적으로 달라집니다.

**필수 조건**:
1. 타겟 MCU의 **Core Power Rail** 정확히 식별
2. 해당 레일의 **Decoupling Capacitor 대량 제거** (가장 중요!)
3. Husky Glitch 출력과 Power Rail 사이 **저인덕턴스 배선**
4. External Clean Power Supply 사용 권장

이 노트북에서는 이를 **최대한 상세히** 설명합니다.


---

# 🔌 3단계 — Voltage Glitching을 위한 상세 하드웨어 배선 가이드

> **이 단계가 이 노트북에서 가장 중요합니다.**

### 전체 연결 구성도 (Voltage Glitch Wire-Tapping)

```
호스트 PC (Jupyter)
    │
    ├── USB → ChipWhisperer-Lite
    │         ├── UART (TX/RX)  ────────→ Target UART (TX/RX)
    │         └── SWD/JTAG      ────────→ Target 프로그래밍 (STM32F3 예시)
    │
    └── USB → ChipWhisperer-Husky
              ├── Trigger In (TIO4 또는 D0) ←──── Target GPIO (TRIG 핀)
              ├── Glitch HP Out (SMA) ────────→ Target Vcc / Core Power Rail
              │                                   (Decoupling Cap 제거 후 연결)
              ├── (선택) HS2 (clkgen) ────────→ Target CLKIN (Husky가 클럭 공급할 경우)
              └── (선택) Measure SMA ←──── Target Shunt Resistor (SCA 확장용)
```

### 상세 배선 설명 및 추천 방법

#### 1. Lite 연결 (변경 없음)
- **UART**: Lite의 TIO1/TIO2 또는 20-pin 커넥터의 UART 핀 → Target UART
- **프로그래밍**: Lite의 SWD/JTAG → Target SWD (STM32F3의 경우)

#### 2. Husky Trigger 연결 (필수)
- Husky **TIO4** (또는 Trigger In) ← Target의 **GPIO 핀** (펌웨어에서 `trigger_high()` / `trigger_low()` 호출)
- FIA 데모 펌웨어에서는 보통 `trigger_high()`를 for-loop 앞에 넣음

#### 3. Husky Voltage Glitch 연결 (가장 중요!)

**Husky의 Glitch 출력**:
- Husky는 **Glitch HP** (High Power MOSFET)와 **Glitch LP** (Low Power MOSFET)를 지원
- Voltage Glitching에는 주로 **Glitch HP** 사용 (더 강력한 단락)

**배선 방법 (추천 순서)**:

1. **타겟 보드 Power Rail 확인**
   - MCU Datasheet에서 **VDD / VCORE / AVDD** 등 Core Power 핀 확인
   - PCB 레이아웃 또는 스키매틱에서 해당 핀의 trace 확인

2. **Decoupling Capacitor 제거 (강력 추천)**
   - MCU 전원 핀 근처의 **모든 Decoupling Cap (0.1uF, 10uF 등)** 제거
   - 이유: Capacitor가 글리치(전압 강하)를 "먹어버림"
   - 제거 후 글리치 성공률이 dramatically 상승

3. **Glitch 연결 방법**
   - **추천**: Husky의 **Glitch SMA** → SMA 케이블 → Target Power Rail에 직접 연결
   - 또는 CW313 보드를 통해 연결
   - **와이어 사용 시**: 가능한 한 **짧고 굵은 와이어** 사용 (인덕턴스 최소화)
   - 연결 지점: MCU의 Vcc 핀 **직접** 또는 Power Rail trace

4. **External Power Supply 사용 (강력 추천)**
   - 타겟을 **Husky나 Lite의 내장 전원**이 아닌 **외부 깨끗한 전원**으로 공급
   - 이유: 글리치 시 전류가 순간적으로 크게 흐르기 때문에 내장 전원이 불안정해질 수 있음
   - Riden RK6006, programmable PSU 등 사용

#### 4. 추가 연결 (선택)
- Husky HS2 (`clkgen`) → Target CLKIN (Husky가 클럭을 공급하면서 글리치)
- Measure Pos SMA → Shunt Resistor (추후 SCA와 결합 시)

### ⚠️ Voltage Glitching 안전 주의사항

- **Decoupling Cap 제거 없이** 글리치하면 성공률이 매우 낮음
- 너무 강한 글리치 (넓은 width + HP) → 타겟 **영구 손상** 또는 **브릭** 가능성
- External PSU 사용 시 전류 제한을 걸어두는 것이 안전
- 처음에는 **매우 좁은 범위**로 테스트 (width 1~10, offset 작은 값부터)
- Husky의 `glitch_hp`를 켜면 강력하니 주의


---

# 📦 1단계 — 라이브러리 임포트 및 다중 장치 연결


In [ ]:
import chipwhisperer as cw
import numpy as np
from tqdm.notebook import trange, tqdm
import time
from collections import Counter

print("ChipWhisperer 버전:", cw.__version__)

### 1.1 다중 장치 연결 함수

In [ ]:
def connect_all_devices():
    """시리얼 넘버 기반으로 Lite와 Husky 연결"""
    device_list = cw.list_devices()
    if not device_list:
        raise RuntimeError("연결된 ChipWhisperer 장치가 없습니다!")

    print(f"발견된 장치 수: {len(device_list)}\n")
    scopes = {}
    for device in device_list:
        name = device['name'].replace("-", "_")
        sn = device['sn']
        try:
            scopes[name] = cw.scope(sn=sn)
            print(f"  [✓] {name} 연결 완료 (SN: {sn})")
        except Exception as e:
            print(f"  [✗] {name} 연결 실패: {e}")
    return scopes

scopes = connect_all_devices()
lite_scope = scopes.get("ChipWhisperer_Lite")
husky_scope = scopes.get("ChipWhisperer_Husky")

if lite_scope is None or husky_scope is None:
    raise RuntimeError("Lite와 Husky가 모두 연결되어 있어야 합니다!")

---

# 🔌 2단계 — Lite를 통한 타겟 프로그래밍 및 UART 연결


In [ ]:
PLATFORM = 'CW308_STM32F3'
SS_VER = 'SS_VER_2_1'

target_type = cw.targets.SimpleSerial2 if SS_VER == "SS_VER_2_1" else cw.targets.SimpleSerial
target = cw.target(lite_scope, target_type)
print("[✓] Lite를 통해 타겟 연결 완료")

### 펌웨어 빌드 및 프로그래밍 (Lite 사용)

In [ ]:
import subprocess
import os

FIRMWARE_DIR = "simpleserial-main"
print("펌웨어 컴파일 중...")

result = subprocess.run(
    ["make", f"PLATFORM={PLATFORM}", "CRYPTO_TARGET=NONE", f"SS_VER={SS_VER}"],
    cwd=FIRMWARE_DIR, capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("펌웨어 컴파일 실패")

print("펌웨어 컴파일 완료")

prog = cw.programmers.STM32FProgrammer
lite_scope.default_setup()
cw.program_target(lite_scope, prog, os.path.join(FIRMWARE_DIR, f"simpleserial-base-{PLATFORM}.hex"))
print("[✓] 타겟 프로그래밍 완료 (Lite 경유)")

---

# ⚙️ 4단계 — Husky Voltage Glitch 모듈 설정


### Husky Voltage Glitch 설정 함수

In [ ]:
def husky_voltage_glitch_setup(scope):
    """
    Husky를 Voltage Glitching 모드로 설정
    - glitch_only 출력 모드
    - High Power MOSFET (glitch_hp) 활성화
    - ext_single 트리거
    """
    scope.glitch.clk_src = "clkgen"
    scope.glitch.output = "glitch_only"      # Voltage Glitch에 최적
    scope.glitch.trigger_src = "ext_single"  # 타겟 GPIO 트리거
    scope.glitch.repeat = 1

    # Husky Voltage Glitch 전용 설정
    scope.io.glitch_hp = True                 # High Power Crowbar 활성화 (강력 글리치)
    scope.io.glitch_lp = False                # Low Power는 필요시 True

    # 클럭 설정 (필요시)
    scope.clock.adc_mul = 1
    scope.clock.clkgen_freq = 7_372_800

    # IO 설정
    scope.io.hs2 = "clkgen"                   # (선택) Husky가 클럭 공급
    scope.io.tio1 = "serial_rx"
    scope.io.tio2 = "serial_tx"

    print("[✓] Husky Voltage Glitch 모듈 설정 완료")
    print(f"   - output       : {scope.glitch.output}")
    print(f"   - glitch_hp    : {scope.io.glitch_hp}")
    print(f"   - trigger_src  : {scope.glitch.trigger_src}")

    return scope

husky_scope = husky_voltage_glitch_setup(husky_scope)

---

# ✅ 5단계 — 베이스라인 통신 테스트


In [ ]:
def my_fsr_cmd(target, cmd, scmd, data, payload_only=False):
    target.simpleserial_write(cmd, data)
    time.sleep(0.05)
    response = target.simpleserial_read(scmd, 16)
    return response

print("베이스라인 통신 테스트 중...")
try:
    resp = my_fsr_cmd(target, 'p', 'r', bytearray([0x00]*16))
    print(f"응답: {resp.hex() if resp else 'None'}")
    expected_ret = resp
    print("[✓] 베이스라인 통신 성공")
except Exception as e:
    print(f"[✗] 통신 실패: {e}")
    expected_ret = None

---

# 🔍 6단계 — Voltage Glitch 파라미터 광범위 탐색


### 결과 분류 함수 (동일)

In [ ]:
def classify_glitch_result(response, expected, timeout=False):
    if timeout or response is None:
        return 0, "freezing"
    if len(response) == 0:
        return 0, "freezing"

    if len(response) < len(expected):
        return 2, "for-loop skip"

    faults = sum(1 for a, b in zip(response, expected) if a != b)
    if faults == 0:
        return 1, "normal"
    elif faults == 1:
        return 3, "one faulty byte"
    elif 2 <= faults <= 4:
        return 4, "few faulty bytes"
    else:
        return 5, "etc (>=5 faulty bytes)"

### Voltage Glitch 탐색 루프

In [ ]:
# Voltage Glitch용 탐색 범위 (처음에는 좁게!)
EXT_OFFSET_RANGE = range(0, 100, 10)
OFFSET_RANGE     = range(0, 300, 30)   # Husky 고해상도
WIDTH_RANGE      = range(1, 80, 10)

results = []
print("Voltage Glitch 파라미터 탐색 시작...")

for ext_off in tqdm(EXT_OFFSET_RANGE, desc="ext_offset"):
    for off in OFFSET_RANGE:
        for wid in WIDTH_RANGE:
            husky_scope.glitch.ext_offset = ext_off
            husky_scope.glitch.offset = off
            husky_scope.glitch.width = wid

            husky_scope.arm()

            try:
                resp = my_fsr_cmd(target, 'p', 'r', bytearray([0x00]*16))
                code, label = classify_glitch_result(resp, expected_ret)
            except Exception:
                code, label = 0, "freezing"

            results.append([code, ext_off, wid, off, label])

print(f"\n총 {len(results)}개 조합 탐색 완료")

In [ ]:
vglitch_result_arr = np.array(results, dtype=object)
print("결과 배열 shape:", vglitch_result_arr.shape)

code_counts = Counter(vglitch_result_arr[:, 0])
print("\n결과 코드 분포:")
for code in sorted(code_counts.keys()):
    print(f"  코드 {code}: {code_counts[code]}회")

---

# 📊 7단계 — 통계 분석으로 최적 파라미터 도출


In [ ]:
from scipy import stats

success_mask = np.isin(vglitch_result_arr[:, 0], [2, 3])
success_results = vglitch_result_arr[success_mask]

if len(success_results) > 0:
    offsets = np.array(success_results[:, 3], dtype=float)
    widths  = np.array(success_results[:, 2], dtype=float)

    mode_offset = stats.mode(offsets, keepdims=True).mode[0]
    mode_width  = stats.mode(widths, keepdims=True).mode[0]

    print(f"✅ 성공 사례 최빈 파라미터 (Voltage Glitch)")
    print(f"   offset (mode): {mode_offset}")
    print(f"   width  (mode): {mode_width}")
    print(f"   성공 사례 수: {len(success_results)} / {len(vglitch_result_arr)}")
else:
    print("⚠️ 성공 사례가 없습니다. 탐색 범위를 조정하거나 Decoupling Cap 제거를 확인하세요.")

---

# 📈 8단계 — Bokeh 시각화


In [ ]:
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.io import output_notebook
output_notebook()

COLOR_MAP = {
    0: '#888888', 1: '#1f77b4', 2: '#2ca02c',
    3: '#d62728', 4: '#ff7f0e', 5: '#9467bd'
}
LABELS = {
    0: "freezing", 1: "normal", 2: "for-loop skip",
    3: "one faulty byte", 4: "few faulty bytes", 5: "etc (>=5 faults)"
}

p = figure(
    width=900, height=500,
    title="Voltage Glitch Parameter Map (Husky Wire-Tap FIA)",
    x_axis_label="offset (1-clock 내 시작 위상)",
    y_axis_label="width (글리치 펄스 폭)",
    tools="pan,wheel_zoom,box_zoom,reset,save,hover",
    active_scroll="wheel_zoom",
    background_fill_color="#fafafa"
)

for code in sorted(set(vglitch_result_arr[:, 0])):
    mask = vglitch_result_arr[:, 0] == code
    if not np.any(mask): continue
    src = ColumnDataSource(data=dict(
        x=np.array(vglitch_result_arr[mask, 3], dtype=float),
        y=np.array(vglitch_result_arr[mask, 2], dtype=float),
        ext=np.array(vglitch_result_arr[mask, 1], dtype=float),
        code=[int(code)] * int(mask.sum()),
        label=[LABELS.get(int(code), str(code))] * int(mask.sum())
    ))
    p.scatter('x', 'y', source=src, size=6, alpha=0.6,
              color=COLOR_MAP.get(int(code), '#000000'),
              legend_label=f"[{code}] {LABELS.get(int(code), code)} (n={int(mask.sum())})")

p.legend.location = "top_right"
p.legend.click_policy = "hide"
hover = HoverTool(tooltips=[("result", "@label"), ("offset", "@x{0}"), ("width", "@y{0}"), ("ext_offset", "@ext{0}")])
p.add_tools(hover)
show(p)

---

# 🔚 9단계 — 장비 연결 해제


In [ ]:
def disconnect_all():
    try:
        target.dis()
        print("[✓] target 연결 해제")
    except: pass
    try:
        lite_scope.dis()
        print("[✓] Lite 연결 해제")
    except: pass
    try:
        husky_scope.dis()
        print("[✓] Husky 연결 해제")
    except: pass

disconnect_all()
print("\n✅ 모든 장치 연결 해제 완료")

---

## 📝 본 노트북 요약 (Voltage Glitching 버전)

| 단계 | 핵심 내용 | 비고 |
|------|-----------|------|
| 1 | 다중 장치 연결 | Lite + Husky |
| 2 | Lite로 프로그래밍 + UART | 통신 전담 |
| 3 | **상세 Voltage Glitch 배선 가이드** | Decap 제거 + Glitch HP 연결 (가장 중요) |
| 4 | `husky_voltage_glitch_setup()` | `glitch_only` + `glitch_hp=True` |
| 5 | Baseline 테스트 | expected_ret 확보 |
| 6 | 3중 루프 탐색 | Voltage Glitch 파라미터 스윕 |
| 7 | 통계 분석 | 최적 (offset, width) 도출 |
| 8 | Bokeh 시각화 | 결과 분포 확인 |
| 9 | 안전한 종료 | disconnect_all() |

### ✅ Voltage Glitching 핵심 학습 포인트

1. **Decoupling Capacitor 제거**가 Voltage Glitch 성공의 핵심
2. **Husky Glitch HP**를 활용한 강력한 크로바 글리치
3. **저인덕턴스 배선**의 중요성
4. External PSU 사용으로 안정성 확보
5. 역할 분리 (Lite 통신 + Husky 공격) 유지

---

**다음 단계 제안**
- 성공 파라미터로 반복 공격 → DFA (Differential Fault Analysis)
- Voltage Glitch + Power Trace 결합 하이브리드 공격
- 실제 제품 보드 (STM32 Nucleo, ESP32 DevKit 등) 적용 실험

*ChipWhisperer Husky — Voltage Glitching Wire-Tapping FIA 노트북 끝*